# 02 — Exploratory Data Analysis
Understand demand behavior before building any forecast model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/processed_sales.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

## 1. Dataset Overview

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())
print('Missing:\n', df.isnull().sum())
print('Stores:', df['store'].nunique(), '| Items:', df['item'].nunique())

## 2. Overall Daily Sales Trend

In [ ]:
daily = df.groupby('date')['sales'].sum().reset_index()

plt.figure(figsize=(12, 4))
plt.plot(daily['date'], daily['sales'])
plt.title('Total Daily Sales — All Stores & Items')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.tight_layout()
plt.show()

**Observations:**
- Sales show a clear repeating seasonal pattern each year.
- There is a slight upward trend over the full period.
- No sudden drops or data gaps visible.

## 3. Weekly Seasonality

In [ ]:
df['day_of_week'] = df['date'].dt.day_name()

dow = df.groupby('day_of_week')['sales'].mean().reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
)

plt.figure(figsize=(8, 4))
dow.plot(kind='bar', color='steelblue')
plt.title('Average Sales by Day of Week')
plt.ylabel('Avg Sales')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Observations:**
- Weekend sales are higher than weekday sales.
- This weekly cycle must be captured in the forecast model.

## 4. Monthly Seasonality

In [ ]:
df['month'] = df['date'].dt.month

monthly = df.groupby('month')['sales'].mean()

plt.figure(figsize=(8, 4))
monthly.plot(kind='bar', color='teal')
plt.title('Average Sales by Month')
plt.ylabel('Avg Sales')
plt.xlabel('Month')
plt.tight_layout()
plt.show()

**Observations:**
- Mid-year and late-year months show higher average sales.
- Seasonality is consistent across years — predictable pattern.

## 5. Demand Distribution

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['sales'], bins=50, color='coral')
plt.title('Distribution of Daily Sales per Store-Item')
plt.xlabel('Daily Sales')
plt.tight_layout()
plt.show()

**Observations:**
- Distribution is right-skewed — most days have moderate sales with occasional high-demand spikes.
- This skewness justifies using safety stock to buffer against demand variability.

## 6. Top & Bottom SKUs by Total Sales

In [ ]:
sku_totals = df.groupby('item')['sales'].sum().sort_values(ascending=False)

print('Top 5 SKUs:')
print(sku_totals.head())
print('\nBottom 5 SKUs:')
print(sku_totals.tail())

sku_totals.head(10).plot(kind='bar', figsize=(10, 4), color='mediumseagreen')
plt.title('Top 10 SKUs by Total Sales')
plt.ylabel('Total Sales')
plt.tight_layout()
plt.show()

**Observations:**
- Top 10 SKUs drive a disproportionate share of total revenue (Pareto effect).
- Low-performing SKUs may warrant different reorder strategies.

## 7. Single SKU Deep Dive — Item 1

In [ ]:
sku1 = df[df['item'] == 1].groupby('date')['sales'].sum().reset_index()

plt.figure(figsize=(12, 4))
plt.plot(sku1['date'], sku1['sales'])
plt.title('Item 1 — Aggregated Daily Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()

## Key Business Insights

1. **Weekly seasonality** — weekend demand peaks consistently; reorder timing should account for this.
2. **Monthly seasonality** — mid-year and year-end spikes require higher safety stock in those periods.
3. **Right-skewed distribution** — occasional demand spikes make safety stock essential.
4. **Pareto SKU structure** — top 10 items should be prioritized for forecast-based reordering.
5. **Upward trend** — a trend-aware forecast model (e.g. Prophet) will outperform simple averages.